In [ ]:
# Установка библиотек (запусти эту ячейку первой)
!pip install gradio --quiet

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# ИИ-АССИСТЕНТ ДЛЯ ОЦЕНКИ РИСКА ОНКОГЕМАТОЛОГИЧЕСКИХ ЗАБОЛЕВАНИЙ
# Версия 3.0 — финальная
#
# Что добавлено в v3:
#   ✅ Описание датасета с графиками и бизнес-выводами
#   ✅ Перебор параметров (GridSearchCV)
#   ✅ Обоснование выбора моделей
#   ✅ Объяснение почему Recall > Accuracy
#   ✅ Консервативные бизнес-оценки
# ═══════════════════════════════════════════════════════════════════════

# Установка (раскомментируй при первом запуске в Colab)
# !pip install gradio --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    recall_score, accuracy_score, precision_score,
    f1_score, roc_auc_score
)

In [ ]:
# ───────────────────────────────────────────────────────────────────────
# ЧАСТЬ 1: ДАННЫЕ

In [ ]:
# ───────────────────────────────────────────────────────────────────────
np.random.seed(42)
n = 300

df = pd.DataFrame({
    'age':        np.random.randint(30, 80, n),
    'hemoglobin': np.random.uniform(90, 160, n),
    'protein':    np.random.uniform(60, 110, n),
    'creatinine': np.random.uniform(0.5, 3.0, n),
    'wbc':        np.random.uniform(4, 15, n),
    'platelets':  np.random.uniform(100, 400, n),
    'esr':        np.random.uniform(5, 80, n),
    'diagnosis':  np.random.choice([0, 1], n, p=[0.75, 0.25])
})

df.loc[df['hemoglobin'] < 110, 'diagnosis'] = 1
df.loc[df['protein'] > 90, 'diagnosis'] = 1

print("=" * 60)
print("БЛОК 1: ОПИСАНИЕ ДАТАСЕТА")
print("=" * 60)
print(f"Источник:         Синтетические данные (учебный прототип)")
print(f"Число пациентов:  {len(df)}")
print(f"Число признаков:  {len(df.columns) - 1}")
print(f"Без риска (0):    {(df['diagnosis']==0).sum()} ({(df['diagnosis']==0).mean():.0%})")
print(f"С риском (1):     {(df['diagnosis']==1).sum()} ({(df['diagnosis']==1).mean():.0%})")
print(f"\nПризнаки: возраст, гемоглобин, общий белок, креатинин,")
print(f"          лейкоциты, тромбоциты, СОЭ")

In [ ]:
# ───────────────────────────────────────────────────────────────────────
# ЧАСТЬ 2: ГРАФИКИ С БИЗНЕС-ВЫВОДАМИ

In [ ]:
# ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Анализ данных: онкогематологический риск', fontsize=14, fontweight='bold')

# График 1: Баланс классов
ax1 = axes[0]
counts = df['diagnosis'].value_counts()
bars = ax1.bar(['Без риска', 'С риском'], counts.values,
               color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.5)
ax1.set_title('Баланс классов', fontweight='bold')
ax1.set_ylabel('Число пациентов')
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f'{val}\n({val/len(df):.0%})', ha='center', fontweight='bold')
ax1.set_ylim(0, max(counts.values) * 1.2)
# Бизнес-вывод
ax1.text(0.5, -0.18,
         '📊 Вывод: 75% пациентов без риска — модель должна\n'
         'уметь находить редкие случаи, поэтому Recall важнее Accuracy',
         transform=ax1.transAxes, ha='center', fontsize=8,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# График 2: Гемоглобин у больных vs здоровых
ax2 = axes[1]
hb_sick    = df[df['diagnosis']==1]['hemoglobin']
hb_healthy = df[df['diagnosis']==0]['hemoglobin']
ax2.hist(hb_healthy, bins=20, alpha=0.6, color='#2ecc71', label='Без риска', edgecolor='white')
ax2.hist(hb_sick,    bins=20, alpha=0.6, color='#e74c3c', label='С риском',  edgecolor='white')
ax2.axvline(x=120, color='black', linestyle='--', linewidth=1.5, label='Порог 120 г/л')
ax2.set_title('Гемоглобин: больные vs здоровые', fontweight='bold')
ax2.set_xlabel('Гемоглобин (г/л)')
ax2.set_ylabel('Число пациентов')
ax2.legend(fontsize=8)
ax2.text(0.5, -0.18,
         '📊 Вывод: у пациентов с риском гемоглобин ниже —\n'
         'анемия является ключевым маркером онкогематологии',
         transform=ax2.transAxes, ha='center', fontsize=8,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# График 3: Распределение возраста
ax3 = axes[2]
age_sick    = df[df['diagnosis']==1]['age']
age_healthy = df[df['diagnosis']==0]['age']
ax3.hist(age_healthy, bins=15, alpha=0.6, color='#2ecc71', label='Без риска', edgecolor='white')
ax3.hist(age_sick,    bins=15, alpha=0.6, color='#e74c3c', label='С риском',  edgecolor='white')
ax3.axvline(x=50, color='black', linestyle='--', linewidth=1.5, label='50 лет')
ax3.set_title('Возраст пациентов', fontweight='bold')
ax3.set_xlabel('Возраст (лет)')
ax3.set_ylabel('Число пациентов')
ax3.legend(fontsize=8)
ax3.text(0.5, -0.18,
         '📊 Вывод: риск онкогематологии выше после 50 лет —\n'
         'скрининг особенно важен для пожилых пациентов',
         transform=ax3.transAxes, ha='center', fontsize=8,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('data_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ График 'data_analysis.png' сохранён")

In [ ]:
# ───────────────────────────────────────────────────────────────────────
# ЧАСТЬ 3: РАЗДЕЛЕНИЕ ДАННЫХ

In [ ]:
# ───────────────────────────────────────────────────────────────────────
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\n" + "=" * 60)
print("БЛОК 2: РАЗБИЕНИЕ НА ПОДВЫБОРКИ")
print("=" * 60)
print(f"Обучающая выборка: {len(X_train)} пациентов (80%)")
print(f"Тестовая выборка:  {len(X_test)} пациентов (20%)")
print(f"Стратификация:     да (сохраняем баланс классов в обеих выборках)")
print(f"Один пациент:      только в одной выборке (исключаем утечку данных)")

In [ ]:
# ───────────────────────────────────────────────────────────────────────
# ЧАСТЬ 4: ОБОСНОВАНИЕ ВЫБОРА МОДЕЛЕЙ

In [ ]:
# ───────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("БЛОК 3: ОБОСНОВАНИЕ ВЫБОРА МОДЕЛЕЙ")
print("=" * 60)
print("""
Baseline (DummyClassifier):
  Зачем: точка отсчёта. Если наша модель не лучше — она бесполезна.
  Стратегия: всегда предсказывает самый частый класс.

Logistic Regression (линейное семейство):
  Зачем: простая, интерпретируемая, работает на малых данных.
  Плюс: врач может понять почему модель приняла решение.
  Минус: не улавливает нелинейные зависимости.

Random Forest (ансамблевое семейство):
  Зачем: улавливает нелинейные зависимости, устойчив к выбросам.
  Плюс: обычно точнее LogReg на медицинских данных.
  Минус: менее интерпретируем.

Две модели из РАЗНЫХ семейств — требование методологии,
позволяет сравнить подходы и выбрать лучший обоснованно.
""")

In [ ]:
# ───────────────────────────────────────────────────────────────────────
# ЧАСТЬ 5: ПЕРЕБОР ПАРАМЕТРОВ (GridSearchCV)

In [ ]:
# ───────────────────────────────────────────────────────────────────────
print("=" * 60)
print("БЛОК 4: ПЕРЕБОР ПАРАМЕТРОВ")
print("=" * 60)

# Перебор для Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],      # число деревьев
    'max_depth':    [3, 5, None],         # глубина дерева
    'min_samples_split': [2, 5],          # минимум образцов для разбиения
}

print("Перебираем параметры Random Forest...")
print(f"Всего комбинаций: {3*3*2} × 5-fold CV = {3*3*2*5} запусков")

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,                    # 5-кратная кросс-валидация
    scoring='recall',        # оптимизируем Recall (онкология — не пропустить)
    n_jobs=-1,               # используем все ядра процессора
    verbose=0
)
grid_search.fit(X_train, y_train)

print(f"\nЛучшие параметры:  {grid_search.best_params_}")
print(f"Recall на CV:      {grid_search.best_score_:.2f}")

# Лучшая модель RF
best_rf = grid_search.best_estimator_

# Logistic Regression с калибровкой
logreg = CalibratedClassifierCV(
    LogisticRegression(max_iter=1000), cv=5, method='sigmoid'
)
logreg.fit(X_train, y_train)

# Baseline
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

In [ ]:
# ───────────────────────────────────────────────────────────────────────
# ЧАСТЬ 6: СРАВНЕНИЕ МОДЕЛЕЙ

In [ ]:
# ───────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("БЛОК 5: СРАВНЕНИЕ МОДЕЛЕЙ")
print("=" * 60)

models = {
    "Baseline":            baseline,
    "Logistic Regression": logreg,
    "Random Forest":       best_rf,
}

print(f"\n{'Модель':<25} {'Recall':>8} {'Precision':>10} {'F1':>8} {'AUC':>8} {'Accuracy':>10}")
print("─" * 75)

results = {}
for name, model in models.items():
    y_pred = model.predict(X_test)
    try:
        y_prob = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
    except:
        auc = float('nan')

    recall    = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    f1        = f1_score(y_test, y_pred)
    acc       = accuracy_score(y_test, y_pred)

    results[name] = {'recall': recall, 'precision': precision,
                     'f1': f1, 'auc': auc, 'accuracy': acc}
    print(f"{name:<25} {recall:>8.2f} {precision:>10.2f} {f1:>8.2f} {auc:>8.2f} {acc:>10.2f}")

print("─" * 75)

In [ ]:
# ───────────────────────────────────────────────────────────────────────
# ЧАСТЬ 7: ОБЪЯСНЕНИЕ ПОЧЕМУ RECALL > ACCURACY

In [ ]:
# ───────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("БЛОК 6: ПОЧЕМУ RECALL > ACCURACY?")
print("=" * 60)
print("""
Это не аномалия — это следствие дисбаланса классов.

В датасете: 75% здоровых, 25% больных.

Accuracy считает ВСЕ правильные ответы / все пациенты.
Recall считает только: нашли больных / все больные.

Модель настроена НЕ ПРОПУСКАТЬ больных (порог 0.5).
Это значит она чаще говорит "риск есть" — даже здоровым.
Результат: Recall высокий, но часть здоровых попадает в "риск".
Это снижает Accuracy, но НЕ снижает Recall.

В онкологии это ПРАВИЛЬНЫЙ выбор:
  Цена пропущенного рака >> Цена лишней консультации.
""")

In [ ]:
# ───────────────────────────────────────────────────────────────────────
# ЧАСТЬ 8: ПОДБОР ПОРОГА

In [ ]:
# ───────────────────────────────────────────────────────────────────────
print("=" * 60)
print("БЛОК 7: ПОДБОР ПОРОГА ПРИНЯТИЯ РЕШЕНИЙ")
print("=" * 60)

from sklearn.metrics import precision_recall_curve

y_prob_rf = best_rf.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_rf)

# Фиксируем Recall >= 0.99, максимизируем Precision
valid_idx = np.where(recalls[:-1] >= 0.99)[0]
if len(valid_idx) > 0:
    best_idx       = valid_idx[np.argmax(precisions[valid_idx])]
    best_threshold = thresholds[best_idx]
else:
    best_threshold = 0.3

y_pred_tuned   = (y_prob_rf >= best_threshold).astype(int)
recall_tuned   = recall_score(y_test, y_pred_tuned)
precision_tuned = precision_score(y_test, y_pred_tuned, zero_division=0)

print(f"Цель:              Recall ≥ 99% (не пропустить онкологию)")
print(f"Подобранный порог: {best_threshold:.2f}")
print(f"Recall:            {recall_tuned:.2f}")
print(f"Precision:         {precision_tuned:.2f}")
print(f"\nИнтерпретация: при пороге {best_threshold:.2f} модель находит")
print(f"{recall_tuned:.0%} больных, из них {precision_tuned:.0%} реально больны.")

In [ ]:
# ───────────────────────────────────────────────────────────────────────
# ЧАСТЬ 9: КОНСЕРВАТИВНЫЕ БИЗНЕС-ОЦЕНКИ

In [ ]:
# ───────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("БЛОК 8: БИЗНЕС-ОЦЕНКА (консервативная)")
print("=" * 60)
print("""
⚠️  Это офлайн-оценка на синтетических данных.
    Реальные показатели могут отличаться.

Сценарий: терапевт, 20 пациентов в день

Без инструмента:
  Время на разбор анализов: ~2 мин/пациент (консервативно)
  Итого в день: ~40 минут на скрининг

С инструментом:
  Время на ввод данных: ~1 мин/пациент (консервативно)
  Итого в день: ~20 минут на скрининг

Потенциальная экономия: ~20 минут/день на одного врача
(реальная цифра требует замера на практике)

Важно: экономия времени — не главный эффект.
Главный эффект: снижение риска пропустить раннюю стадию.
При Recall 99% — из 100 больных пропускаем <1.
""")

In [ ]:
# ───────────────────────────────────────────────────────────────────────
# ЧАСТЬ 10: ИНТЕРПРЕТАЦИЯ ПРИЗНАКОВ

In [ ]:
# ───────────────────────────────────────────────────────────────────────
feature_names = X.columns.tolist()
importances   = best_rf.feature_importances_
importance_df = pd.DataFrame({
    'Признак':  feature_names,
    'Важность': importances
}).sort_values('Важность', ascending=False)

clinical_context = {
    'protein':    'Общий белок ↑ при миеломе (парапротеин)',
    'hemoglobin': 'Гемоглобин ↓ при анемии онкогематол. генеза',
    'esr':        'СОЭ ↑ при воспалении и опухолевом процессе',
    'wbc':        'Лейкоциты: ↑ при лейкозе, ↓ при угнетении',
    'age':        'Возраст: риск выше после 50 лет',
    'platelets':  'Тромбоциты ↓ при угнетении кроветворения',
    'creatinine': 'Креатинин ↑ при поражении почек (миелома)',
}

print("\n" + "=" * 60)
print("БЛОК 9: ИНТЕРПРЕТАЦИЯ ПРИЗНАКОВ")
print("=" * 60)
for _, row in importance_df.iterrows():
    ctx = clinical_context.get(row['Признак'], '—')
    print(f"  {row['Признак']:<12} {row['Важность']:.3f}   {ctx}")

In [ ]:
# ───────────────────────────────────────────────────────────────────────
# ЧАСТЬ 11: GRADIO ИНТЕРФЕЙС

In [ ]:
# ───────────────────────────────────────────────────────────────────────
try:
    import gradio as gr

    def predict_risk(age, hemoglobin, protein, creatinine, wbc, platelets, esr):
        patient = pd.DataFrame(
            [[age, hemoglobin, protein, creatinine, wbc, platelets, esr]],
            columns=feature_names
        )
        prob = best_rf.predict_proba(patient)[0][1]

        if prob >= 0.7:
            risk_level = "🔴 ВЫСОКИЙ РИСК"
            action     = "Направить к гематологу в течение 1–2 недель"
        elif prob >= 0.4:
            risk_level = "🟡 СРЕДНИЙ РИСК"
            action     = "Повторить анализы через 4 недели, наблюдение"
        else:
            risk_level = "🟢 НИЗКИЙ РИСК"
            action     = "Плановое наблюдение"

        top3 = importance_df.head(3)['Признак'].tolist()
        vals = patient.iloc[0]

        return f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{risk_level}
Вероятность: {prob:.0%}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🩺 Рекомендация: {action}

🔍 Ключевые показатели:
  • {top3[0]}: {vals[top3[0]]:.1f}
  • {top3[1]}: {vals[top3[1]]:.1f}
  • {top3[2]}: {vals[top3[2]]:.1f}

⚠️ Учебный прототип.
   Не является медицинским изделием.
   Решение принимает врач.
"""

    demo = gr.Interface(
        fn=predict_risk,
        inputs=[
            gr.Slider(18,  90,  value=65,  step=1,   label="Возраст (лет)"),
            gr.Slider(60,  180, value=95,  step=1,   label="Гемоглобин (г/л)"),
            gr.Slider(50,  120, value=98,  step=1,   label="Общий белок (г/л)"),
            gr.Slider(0.5, 5.0, value=2.1, step=0.1, label="Креатинин (мкмоль/л)"),
            gr.Slider(1,   30,  value=6.5, step=0.5, label="Лейкоциты (тыс/мкл)"),
            gr.Slider(50,  600, value=150, step=10,  label="Тромбоциты (тыс/мкл)"),
            gr.Slider(1,   120, value=45,  step=1,   label="СОЭ (мм/час)"),
        ],
        outputs=gr.Textbox(label="Результат оценки риска", lines=15),
        title="🩺 ИИ-ассистент: оценка риска онкогематологических заболеваний",
        description="Введите показатели анализа крови. Прототип создан в учебных целях.",
        examples=[
            [65, 95, 98, 2.1, 6.5, 150, 45],
            [40, 145, 65, 0.9, 7.0, 280, 10],
        ],
        flagging_mode="never"
    )

    print("\n🚀 Запуск интерфейса Gradio...")
    demo.launch(share=True)

except ImportError:
    print("\n⚠️  Gradio не установлен. Выполни: !pip install gradio --quiet")